In [ ]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv
from time import sleep
import base64

# === Load tokens ===
load_dotenv("All_Tokens.env")
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 7)]
tokens = [t for t in tokens if t]
if not tokens:
    raise ValueError("No GitHub tokens found.")

token_index = 0
def get_headers():
    global token_index
    token = tokens[token_index]
    print(f"🔁 Using token #{token_index + 1}")
    token_index = (token_index + 1) % len(tokens)
    return {
        "Authorization": f"token {token}",
        "Accept": "application/vnd.github.v3+json",
        "User-Agent": "android-repo-crawler/1.0"
    }

# === File paths ===
input_path = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step2_verified_output.csv"
output_path = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step3_android_detection_output.csv"

# === Load data ===
df = pd.read_csv(input_path)

# === Init output columns ===
df["android_metadata_match"] = ""
df["android_in_readme"] = ""

def search_android_in_files(repo):
    # 1. Check README
    readme_url = f"https://api.github.com/repos/{repo}/readme"
    r = requests.get(readme_url, headers=get_headers())
    if r.status_code == 200:
        try:
            content = r.json().get("content", "")
            decoded = base64.b64decode(content).decode("utf-8", errors="ignore").lower()
            if "android" in decoded:
                return True
        except:
            pass

    # 2. Check other .md/.rst files
    contents_url = f"https://api.github.com/repos/{repo}/contents"
    r = requests.get(contents_url, headers=get_headers())
    if r.status_code == 200:
        for item in r.json():
            name = item.get("name", "").lower()
            if name.endswith((".md", ".rst", ".markdown")):
                file_url = item.get("download_url")
                if file_url:
                    try:
                        text = requests.get(file_url, headers=get_headers()).text.lower()
                        if "android" in text:
                            return True
                    except:
                        pass

    # 3. Optionally check Wiki (Home.md)
    wiki_url = f"https://raw.githubusercontent.com/wiki/{repo}/Home.md"
    r = requests.get(wiki_url, headers={"User-Agent": "android-repo-crawler/1.0"})
    if r.status_code == 200 and "android" in r.text.lower():
        return True

    return False

# === Process each repo ===
for i, row in df.iterrows():
    if row["Valid_Repo"] != "yes":
        df.at[i, "android_metadata_match"] = "N/A"
        df.at[i, "android_in_readme"] = "N/A"
        continue

    name = str(row.get("name", "")).lower()
    topics = str(row.get("topics", "")).lower()
    desc = str(row.get("description", "")).lower() if "description" in row else ""
    
    metadata_hit = "android" in name or "android" in topics or "android" in desc
    df.at[i, "android_metadata_match"] = "yes" if metadata_hit else "no"

    found = search_android_in_files(row["full_name"])
    df.at[i, "android_in_readme"] = "yes" if found else "no"

    if i % 100 == 0:
        print(f"🔍 Checked {i+1} repos...")

# === Save output ===
df.to_csv(output_path, index=False)
print(f"✅ Step 3 complete. Saved to: {output_path}")
